# EDA 04: Delivery Performance, Freight Costs & Supplier Efficiency

This notebook evaluates freight costs, supplier lead times, reliability scores, and net profit margins across operating regions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

df_sales = pd.read_parquet('data/processed/sales_fact.parquet')
df_supp = pd.read_parquet('data/processed/supplier_dim.parquet')
df_wh = pd.read_parquet('data/processed/warehouse_dim.parquet')

print(f"Sales Records: {len(df_sales):,}, Suppliers: {len(df_supp):,}, Warehouses: {len(df_wh):,}")


Sales Records: 1,143,942, Suppliers: 3,111, Warehouses: 1,206


In [2]:
# Shipping Cost Distribution & Freight Ratio
df_sales['freight_ratio'] = np.where(df_sales['total_sales'] > 0, df_sales['shipping_cost'] / df_sales['total_sales'], 0.0)

freight_stats = df_sales[['shipping_cost', 'freight_ratio', 'profit']].describe().T
print("Shipping Cost and Freight Ratio Statistics:")
print(freight_stats)

plt.figure(figsize=(10, 5))
sns.histplot(df_sales[df_sales['shipping_cost'] > 0]['shipping_cost'], bins=50, kde=True, color='teal')
plt.title('Distribution of Shipping Freight Costs ($)')
plt.xlabel('Shipping Cost ($)')
plt.ylabel('Frequency')
plt.show()


Shipping Cost and Freight Ratio Statistics:
                   count      mean        std  ...  50%  75%         max
shipping_cost  1143942.0  1.968552   7.751124  ...  0.0  0.0  409.680000
freight_ratio  1143942.0  0.031022   0.131333  ...  0.0  0.0   12.134134
profit         1143942.0  3.467748  42.252118  ...  0.0  0.0  911.799988

[3 rows x 8 columns]


In [3]:
# Regional Profit Margin & Shipping Cost Analysis
region_delivery = df_sales.groupby('region').agg(
    total_revenue=('total_sales', 'sum'),
    total_freight=('shipping_cost', 'sum'),
    total_profit=('profit', 'sum'),
    avg_freight=('shipping_cost', 'mean'),
    avg_profit=('profit', 'mean'),
    order_count=('sales_id', 'count')
).reset_index().sort_values(by='total_revenue', ascending=False)

region_delivery['profit_margin_pct'] = np.where(region_delivery['total_revenue'] > 0, (region_delivery['total_profit'] / region_delivery['total_revenue']) * 100, 0.0)
print(region_delivery.head(10))

plt.figure(figsize=(12, 6))
sns.barplot(data=region_delivery.head(10), x='avg_freight', y='region', palette='mako')
plt.title('Top 10 Regions by Average Shipping Freight Cost ($)')
plt.xlabel('Average Freight Cost ($)')
plt.ylabel('Region')
plt.tight_layout()
plt.show()


             region  total_revenue  ...  order_count  profit_margin_pct
42               US   6.710984e+09  ...         6435           0.000000
9                DE   5.851200e+09  ...       844338           0.000000
35               SP   8.421077e+06  ...        80342           0.000000
47   Western Europe   5.676232e+06  ...        27109          11.018685
7   Central America   5.666454e+06  ...        28341          10.877024
36    South America   2.961217e+06  ...        14935          11.318129
22  Northern Europe   2.074089e+06  ...         9792          11.255573
23          Oceania   2.004613e+06  ...        10148          10.050719
41  Southern Europe   1.970845e+06  ...         9431          11.712198
39   Southeast Asia   1.896733e+06  ...         9539          11.142467

[10 rows x 8 columns]


In [4]:
# Supplier Lead Time & Reliability Analysis
supp_merged = df_sales.merge(df_supp.drop(columns=['dataset_source', 'region', 'city', 'country'], errors='ignore'), on='supplier_id', how='left')

supp_perf = supp_merged.groupby(['supplier_id', 'supplier_name', 'lead_time_days', 'reliability_score']).agg(
    order_count=('sales_id', 'count'),
    total_revenue=('total_sales', 'sum'),
    avg_profit=('profit', 'mean')
).reset_index().sort_values(by='order_count', ascending=False)

print("Top Suppliers by Volume & Performance:")
print(supp_perf.head(10))


Top Suppliers by Volume & Performance:
          supplier_id  ... avg_profit
3107  rossmann_supp_a  ...   0.000000
3110  rossmann_supp_d  ...   0.000000
3109  rossmann_supp_c  ...   0.000000
8       dataco_supp_7  ...  27.432366
5       dataco_supp_4  ...  17.998345
6       dataco_supp_5  ...  14.976627
3108  rossmann_supp_b  ...   0.000000
4       dataco_supp_3  ...  28.242513
7       dataco_supp_6  ...  14.996021
11    m5_supp_walmart  ...   0.000000

[10 rows x 7 columns]
